# Breast Cancer Diagnosis — Member A: Data & Distributions

**Author:** MD Rabbi  
**Course:** Statistics — Final Project  
**Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC)  
**Source:** Kaggle — [Breast Cancer Wisconsin (Diagnostic) Data Set](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data) (originally from the UCI Machine Learning Repository)

## My responsibilities (Member A)
1. Load the dataset and document the features
2. Run a data quality check (missing values, duplicates, class balance)
3. Compute descriptive statistics separately for malignant and benign tumors
4. Plot distributions (histograms + KDE) for all features, grouped by diagnosis
5. Plot boxplots for all features comparing malignant vs. benign
6. Hand off a clean dataset and these summaries to Member B (correlation/hypothesis tests) and Member C (feature importance)

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

FIG_DIR = '../figures'
DATA_DIR = '../data'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

PALETTE = {'Malignant': '#D62728', 'Benign': '#1F77B4'}

print('Libraries loaded.')

## 2. Load the dataset (from Kaggle CSV)

**Source:** [Breast Cancer Wisconsin (Diagnostic) Data Set](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data) on Kaggle, originally contributed by Dr. William Wolberg, University of Wisconsin (UCI ML Repository, 1995).

**To run this notebook:** Download `data.csv` from the Kaggle page, rename it to `wdbc.csv`, and place it in `../data/wdbc.csv`.

**About the data:** Each row is a tumor sample. For each sample, 10 properties were measured from a digitized image of a fine-needle aspirate (FNA) of a breast mass. For each base property, three statistics were recorded — the **mean**, the **standard error**, and the **worst** (mean of the three largest values). That gives 10 × 3 = 30 features. Each tumor is labeled either **Malignant (M)** or **Benign (B)**.

**The 10 base measurements:** radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension.

In [ ]:
KAGGLE_CSV = '../data/wdbc.csv'

if not os.path.exists(KAGGLE_CSV):
    raise FileNotFoundError(
        f'Could not find {KAGGLE_CSV}.\n'
        f'Download data.csv from https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data\n'
        f'rename it to wdbc.csv, and place it in the ../data/ folder.'
    )

raw = pd.read_csv(KAGGLE_CSV)
print(f'Raw Kaggle file shape: {raw.shape}')
print(f'Raw columns: {list(raw.columns)}')
raw.head()

### 2a. Clean the Kaggle CSV

The Kaggle CSV has two quirks we need to fix:
1. There's a trailing empty column called `Unnamed: 32` (an export artifact) — drop it.
2. There's an `id` column that isn't useful for analysis — drop it.
3. The `diagnosis` column uses `M`/`B` codes — convert to readable `Malignant`/`Benign` labels.
4. Reorder so `diagnosis` is the last column.

In [ ]:
df = raw.copy()

if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])
if 'id' in df.columns:
    df = df.drop(columns=['id'])

df['diagnosis'] = df['diagnosis'].map({'M': 'Malignant', 'B': 'Benign'})

feature_cols = [c for c in df.columns if c != 'diagnosis']
df = df[feature_cols + ['diagnosis']]

df.to_csv(f'{DATA_DIR}/wdbc_clean.csv', index=False)

print(f'Cleaned shape: {df.shape}')
print(f'Total samples: {len(df)}')
print(f'Total features: {len(feature_cols)} (plus diagnosis label)')
print(f'\nSaved cleaned dataset to: {DATA_DIR}/wdbc_clean.csv')
df.head()

## 3. Data quality check

In [ ]:
missing = df.isnull().sum().sum()
print(f'Total missing values: {missing}')
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
print(f'\nFeature dtypes:')
print(df.dtypes.value_counts())

In [ ]:
class_counts = df['diagnosis'].value_counts()
class_pct = df['diagnosis'].value_counts(normalize=True) * 100
balance_table = pd.DataFrame({'Count': class_counts, 'Percentage': class_pct.round(2)})
print('Class balance:')
print(balance_table)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.index, class_counts.values,
              color=[PALETTE[x] for x in class_counts.index], edgecolor='black')
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Class Balance: Malignant vs. Benign', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of samples')
ax.set_ylim(0, max(class_counts.values) * 1.15)
plt.savefig(f'{FIG_DIR}/01_class_balance.png')
plt.show()

**Observation:** The dataset is moderately imbalanced — about 63% benign, 37% malignant. Member C should keep this in mind for any classifier (use stratified splits and metrics beyond accuracy). For descriptive statistics and hypothesis tests, the imbalance is fine.

## 4. Descriptive statistics — overall and by diagnosis

In [ ]:
feature_cols = [c for c in df.columns if c != 'diagnosis']

def descriptive_stats(group_df, feature_cols):
    rows = []
    for col in feature_cols:
        s = group_df[col]
        rows.append({
            'feature': col, 'mean': s.mean(), 'median': s.median(),
            'std': s.std(), 'min': s.min(), 'max': s.max(),
            'skewness': stats.skew(s), 'kurtosis': stats.kurtosis(s),
        })
    return pd.DataFrame(rows).set_index('feature')

stats_overall = descriptive_stats(df, feature_cols)
stats_malignant = descriptive_stats(df[df['diagnosis']=='Malignant'], feature_cols)
stats_benign = descriptive_stats(df[df['diagnosis']=='Benign'], feature_cols)

stats_overall.round(4).to_csv(f'{DATA_DIR}/stats_overall.csv')
stats_malignant.round(4).to_csv(f'{DATA_DIR}/stats_malignant.csv')
stats_benign.round(4).to_csv(f'{DATA_DIR}/stats_benign.csv')

print('Saved: stats_overall.csv, stats_malignant.csv, stats_benign.csv\n')
print('Overall descriptive statistics (all 569 samples):')
stats_overall.round(3)

In [ ]:
print('Malignant tumors (n=212):')
stats_malignant.round(3)

In [ ]:
print('Benign tumors (n=357):')
stats_benign.round(3)

### Side-by-side comparison: mean and std for each feature

This single table is one of the most useful artifacts of Member A's work — it shows at a glance which features differ most between the two groups. Member B will use this as input for hypothesis tests.

In [ ]:
comparison = pd.DataFrame({
    'mean_malignant': stats_malignant['mean'],
    'mean_benign': stats_benign['mean'],
    'std_malignant': stats_malignant['std'],
    'std_benign': stats_benign['std'],
})
pooled_std = np.sqrt((comparison['std_malignant']**2 + comparison['std_benign']**2) / 2)
comparison['mean_diff'] = comparison['mean_malignant'] - comparison['mean_benign']
comparison['standardized_diff'] = comparison['mean_diff'] / pooled_std
comparison_sorted = comparison.reindex(
    comparison['standardized_diff'].abs().sort_values(ascending=False).index)
comparison_sorted.round(4).to_csv(f'{DATA_DIR}/group_comparison.csv')

print('Features ranked by |standardized mean difference| (Cohen\'s d):')
print('(larger = better separation between malignant and benign)\n')
comparison_sorted.round(3)

**What this tells us already:** The features at the top — typically `concave points_worst`, `perimeter_worst`, `concave points_mean`, `radius_worst`, `perimeter_mean`, and `area_worst` — are where malignant and benign tumors differ most. We expect these to dominate Member C's feature importance analysis too.

## 5. Distribution plots

For each of the 30 features, we overlay malignant vs. benign distributions using histograms with KDE smoothing.

In [ ]:
def plot_distributions(df, features, ncols=5, figsize_per_plot=(3.5, 2.5),
                       title='', filename=None):
    n = len(features); nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize_per_plot[0]*ncols,
                                      figsize_per_plot[1]*nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, feat in enumerate(features):
        ax = axes[i]
        for label, color in PALETTE.items():
            subset = df[df['diagnosis']==label][feat]
            ax.hist(subset, bins=25, alpha=0.5, color=color, label=label,
                    density=True, edgecolor='white', linewidth=0.4)
        for label, color in PALETTE.items():
            subset = df[df['diagnosis']==label][feat]
            kde = stats.gaussian_kde(subset)
            x_range = np.linspace(subset.min(), subset.max(), 200)
            ax.plot(x_range, kde(x_range), color=color, linewidth=1.8)
        ax.set_title(feat, fontsize=10)
        ax.set_xlabel(''); ax.set_ylabel('Density', fontsize=8)
        ax.tick_params(labelsize=7)
        if i == 0: ax.legend(fontsize=8)
    for j in range(n, len(axes)): axes[j].axis('off')
    if title: fig.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    if filename: plt.savefig(f'{FIG_DIR}/{filename}')
    plt.show()

In [ ]:
# Kaggle CSV uses suffix naming: feature_mean, feature_se, feature_worst
mean_features = [c for c in feature_cols if c.endswith('_mean')]
se_features = [c for c in feature_cols if c.endswith('_se')]
worst_features = [c for c in feature_cols if c.endswith('_worst')]
print(f'Mean features ({len(mean_features)}):  {mean_features}')
print(f'SE features ({len(se_features)}):    {se_features}')
print(f'Worst features ({len(worst_features)}): {worst_features}')

In [ ]:
plot_distributions(df, mean_features, title='Distribution of MEAN features by diagnosis',
                   filename='02_dist_mean_features.png')

In [ ]:
plot_distributions(df, se_features, title='Distribution of STANDARD ERROR features by diagnosis',
                   filename='03_dist_se_features.png')

In [ ]:
plot_distributions(df, worst_features, title='Distribution of WORST features by diagnosis',
                   filename='04_dist_worst_features.png')

**Observations:**
- Malignant tumors are consistently shifted toward larger values for size-related features (radius, perimeter, area).
- The two distributions are clearly separated for `concave points` and `concavity`, especially in their `worst` form.
- For `fractal_dimension` and `symmetry`, distributions overlap heavily — these features will have weak discriminative power.
- Several features are visibly right-skewed — Member B should consider this when choosing between t-tests and non-parametric tests.

## 6. Boxplots — malignant vs. benign for each feature

In [ ]:
def plot_boxplots(df, features, ncols=5, figsize_per_plot=(3.0, 2.8),
                  title='', filename=None):
    n = len(features); nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize_per_plot[0]*ncols,
                                      figsize_per_plot[1]*nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, feat in enumerate(features):
        ax = axes[i]
        sns.boxplot(data=df, x='diagnosis', y=feat, ax=ax,
                    palette=PALETTE, hue='diagnosis',
                    order=['Benign', 'Malignant'], legend=False,
                    width=0.55, fliersize=3)
        ax.set_title(feat, fontsize=10)
        ax.set_xlabel(''); ax.set_ylabel('')
        ax.tick_params(labelsize=8)
    for j in range(n, len(axes)): axes[j].axis('off')
    if title: fig.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    if filename: plt.savefig(f'{FIG_DIR}/{filename}')
    plt.show()

In [ ]:
plot_boxplots(df, mean_features, title='Boxplots — MEAN features',
              filename='05_box_mean_features.png')

In [ ]:
plot_boxplots(df, se_features, title='Boxplots — STANDARD ERROR features',
              filename='06_box_se_features.png')

In [ ]:
plot_boxplots(df, worst_features, title='Boxplots — WORST features',
              filename='07_box_worst_features.png')

## 7. Spotlight: top 6 most discriminative features

In [ ]:
top_features = comparison_sorted.index[:6].tolist()
print(f'Top 6 features by |standardized mean difference|:\n  ' + '\n  '.join(top_features))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feat in enumerate(top_features):
    ax = axes[i]
    for label, color in PALETTE.items():
        subset = df[df['diagnosis']==label][feat]
        ax.hist(subset, bins=30, alpha=0.55, color=color, label=label,
                density=True, edgecolor='white')
        kde = stats.gaussian_kde(subset)
        x_range = np.linspace(subset.min(), subset.max(), 200)
        ax.plot(x_range, kde(x_range), color=color, linewidth=2.2)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.set_ylabel('Density')
    if i == 0: ax.legend()
fig.suptitle('Top 6 most discriminative features — distributions',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/08_top6_distributions.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feat in enumerate(top_features):
    ax = axes[i]
    sns.boxplot(data=df, x='diagnosis', y=feat, ax=ax,
                palette=PALETTE, hue='diagnosis',
                order=['Benign', 'Malignant'], legend=False, width=0.5)
    sns.stripplot(data=df, x='diagnosis', y=feat, ax=ax,
                  order=['Benign', 'Malignant'],
                  color='black', size=2.0, alpha=0.4, jitter=0.2)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
fig.suptitle('Top 6 most discriminative features — boxplots',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_top6_boxplots.png')
plt.show()

## 8. Summary of Member A's deliverables

**Files written for the team:**
- `data/wdbc_clean.csv` — clean dataset (569 rows × 31 cols, with `diagnosis` label)
- `data/stats_overall.csv` — descriptive statistics for the whole dataset
- `data/stats_malignant.csv` — descriptive statistics for malignant tumors only
- `data/stats_benign.csv` — descriptive statistics for benign tumors only
- `data/group_comparison.csv` — features ranked by standardized mean difference
- `figures/*.png` — all distribution and boxplot figures, ready for the report

**Key takeaways for the report:**
1. The dataset contains 569 samples (357 benign, 212 malignant), with no missing values or duplicates.
2. Each sample has 30 numeric features from 10 underlying measurements (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension), each captured in three forms: mean, standard error, and worst-case value.
3. Malignant tumors show systematically larger values on size-related features and shape-irregularity features.
4. The features with the strongest separation between malignant and benign are dominated by the `worst` family — particularly `concave points_worst`, `perimeter_worst`, `radius_worst`, and `area_worst`.
5. Several features show right-skewed distributions, which Member B should consider when choosing between parametric and non-parametric tests.

**Handoff for Member B:** Use `data/wdbc_clean.csv`. Start your t-tests with the top of `group_comparison.csv` to confirm the differences are statistically significant.

**Handoff for Member C:** Use the same `wdbc_clean.csv`. The top-6 features here are good candidates to verify against your feature-importance ranking.